# Comptage des occurrences des personnages par tome

Ce notebook :

1. lit le fichier `.ods` des personnages ;
2. construit, pour chaque personnage, une liste de variantes de noms à partir des **3 premières colonnes** :
   - `NOMS HUMAINS`
   - `SECOND NOM`
   - `TROISIEME NOM`
3. prend en compte les combinaisons demandées :
   - `1 + 2`
   - `2 + 3`
   - `1 + 2 + 3`
   - `1 + 3`
4. compte les occurrences de chaque personnage dans chaque **tome** (`book`) du fichier `.csv` ;
5. exporte un tableau final qui reprend la structure du tableau `.ods` et ajoute une colonne par tome.

## Remarque importante
Par défaut, le notebook compte aussi les **formes simples** (`1`, `2`, `3`) quand elles existent.  
C'est pratique pour éviter que les personnages à nom simple aient toujours `0`.

Si tu veux **strictement** ne compter que les combinaisons demandées, mets :

```python
INCLUDE_SINGLE_NAMES = False
```
dans la cellule de paramètres.


In [1]:
# Paramètres
ODS_PATH = "../Noms personnages.ods"
CSV_PATH = "../Texte/La-ballade-de-Pern-intégrale-sentences-lemma.csv"
ODS_SHEET = "Noms et genres persos"

# True = compte aussi les noms simples (recommandé)
# False = ne compte QUE 1+2, 2+3, 1+2+3, 1+3
INCLUDE_SINGLE_NAMES = True

# Colonne texte à analyser dans le CSV
TEXT_COLUMN = "text"

# Fichiers de sortie
OUTPUT_XLSX = "pern_occurrences_par_tome.xlsx"
OUTPUT_CSV = "pern_occurrences_par_tome.csv"
OUTPUT_ALIASES_CSV = "pern_aliases_utilises.csv"


In [2]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

In [3]:
# Lecture des fichiers
names_df = pd.read_excel(ODS_PATH, sheet_name=ODS_SHEET, engine="odf")
text_df = pd.read_csv(CSV_PATH)

print("Dimensions ODS :", names_df.shape)
print("Dimensions CSV :", text_df.shape)
print("\nColonnes ODS :", list(names_df.columns))
print("\nColonnes CSV :", list(text_df.columns))

Dimensions ODS : (1026, 7)
Dimensions CSV : (132850, 7)

Colonnes ODS : ['Noms humains', 'Second nom', 'Troisième nom', 'Quatrième nom', 'H/F/Na', 'ID unique', 'Morts']

Colonnes CSV : ['sentence_id', 'book', 'parution', 'start_pdf_page', 'end_pdf_page', 'text', 'text_lemma']


In [4]:
# Vérifications minimales
required_name_cols = ["Noms humains", "Second nom", "Troisième nom", "Quatrième nom", "ID unique"]
required_csv_cols = ["book", TEXT_COLUMN]

missing_ods = [c for c in required_name_cols if c not in names_df.columns]
missing_csv = [c for c in required_csv_cols if c not in text_df.columns]

if missing_ods:
    raise ValueError(f"Colonnes manquantes dans le ODS : {missing_ods}")

if missing_csv:
    raise ValueError(f"Colonnes manquantes dans le CSV : {missing_csv}")

print("Vérification OK.")

Vérification OK.


In [5]:
def clean_part(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


def build_aliases(row, include_single_names=True):
    """Construit les variantes d'un personnage à partir des 3 premières colonnes."""
    p1 = clean_part(row["Noms humains"])
    p2 = clean_part(row["Second nom"])
    p3 = clean_part(row["Troisième nom"])
    p4 = clean_part(row["Quatrième nom"])

    aliases = []

    # Optionnel : formes simples
    if include_single_names:
        for p in (p1, p2, p3):
            if p:
                aliases.append(p)

    # Combinaisons demandées
    if p1 and p2:
        aliases.append(f"{p1} {p2}")      # 1 + 2
    if p2 and p3:
        aliases.append(f"{p2} {p3}")      # 2 + 3
    if p1 and p2 and p3:
        aliases.append(f"{p1} {p2} {p3}") # 1 + 2 + 3
    if p1 and p3:
        aliases.append(f"{p1} {p3}")      # 1 + 3

    # Déduplication en conservant l'ordre
    seen = set()
    clean_aliases = []
    for a in aliases:
        a = re.sub(r"\s+", " ", a).strip()
        key = a.lower()
        if key not in seen:
            seen.add(key)
            clean_aliases.append(a)

    # Important : les alias longs d'abord pour éviter le double comptage
    clean_aliases = sorted(clean_aliases, key=lambda x: (-len(x), x.lower()))
    return clean_aliases


names_df["aliases"] = names_df.apply(
    lambda row: build_aliases(row, include_single_names=INCLUDE_SINGLE_NAMES),
    axis=1
)

names_df[["Noms humains", "Second nom", "Troisième nom", "Quatrième nom", "ID unique", "aliases"]].head(15)

,Noms humains,Second nom,Troisième nom,Quatrième nom,ID unique,aliases
0,Adessa,NaN,NaN,NaN,6,[Adessa]
1,Armald,NaN,NaN,NaN,33,[Armald]
2,B’ner,NaN,NaN,NaN,49,[B’ner]
3,Barly,NaN,NaN,NaN,67,[Barly]
4,Berchar,Berch,NaN,NaN,88,"[Berchar Berch, Berchar, Berch]"
5,Betrice,NaN,NaN,NaN,96,[Betrice]
6,Brare,NaN,NaN,NaN,119,[Brare]
7,C’gan,NaN,NaN,NaN,143,[C’gan]
8,Caesar,Galliani,NaN,NaN,151,"[Caesar Galliani, Galliani, Caesar]"
9,Carola,NaN,NaN,NaN,162,[Carola]


In [6]:
# Tableau d'audit des alias réellement utilisés
aliases_audit = names_df[["ID unique", "Noms humains", "Second nom", "Troisième nom", "Quatrième nom", "ID unique", "aliases"]].copy()
aliases_audit["aliases"] = aliases_audit["aliases"].apply(lambda x: " | ".join(x))
aliases_audit.to_csv(OUTPUT_ALIASES_CSV, index=False)
print(f"Fichier alias exporté : {OUTPUT_ALIASES_CSV}")
aliases_audit.head(10)

Fichier alias exporté : pern_aliases_utilises.csv


,ID unique,Noms humains,Second nom,Troisième nom,Quatrième nom,ID unique,aliases
0,6,Adessa,NaN,NaN,NaN,6,Adessa
1,33,Armald,NaN,NaN,NaN,33,Armald
2,49,B’ner,NaN,NaN,NaN,49,B’ner
3,67,Barly,NaN,NaN,NaN,67,Barly
4,88,Berchar,Berch,NaN,NaN,88,Berchar Berch | Berchar | Berch
5,96,Betrice,NaN,NaN,NaN,96,Betrice
6,119,Brare,NaN,NaN,NaN,119,Brare
7,143,C’gan,NaN,NaN,NaN,143,C’gan
8,151,Caesar,Galliani,NaN,NaN,151,Caesar Galliani | Galliani | Caesar
9,162,Carola,NaN,NaN,NaN,162,Carola


In [7]:
# Préparation des textes par tome
text_df = text_df.copy()
text_df = text_df[text_df["book"].notna()].copy()
text_df["book"] = text_df["book"].astype(int)

books = sorted(text_df["book"].unique())
print("Tomes détectés :", books)

book_texts = (
    text_df.groupby("book")[TEXT_COLUMN]
    .apply(lambda s: "\n".join(s.fillna("").astype(str)))
    .to_dict()
)

for b in books[:5]:
    print(f"Tome {b}: {len(book_texts[b]):,} caractères")

Tomes détectés : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]
Tome 1: 777,188 caractères
Tome 2: 562,067 caractères
Tome 3: 634,318 caractères
Tome 4: 727,919 caractères
Tome 5: 240,296 caractères


In [8]:
def compile_character_pattern(aliases):
    """Crée une regex qui repère les alias du personnage avec bornes de mots."""
    if not aliases:
        return None

    escaped = [re.escape(a) for a in aliases]
    pattern = r"(?<!\w)(?:" + "|".join(escaped) + r")(?!\w)"
    return re.compile(pattern, flags=re.IGNORECASE)


def count_occurrences_in_text(pattern, text):
    if pattern is None:
        return 0
    return len(pattern.findall(text))

In [ ]:
# Comptage des occurrences par personnage et par tome
count_data = defaultdict(list)

for i, row in names_df.iterrows():
    aliases = row["aliases"]
    pattern = compile_character_pattern(aliases)

    for book in books:
        n = count_occurrences_in_text(pattern, book_texts[book])
        count_data[f"book_{book}"].append(n)

    if (i + 1) % 100 == 0:
        print(f"{i + 1} personnages traités / {len(names_df)}")

print("Comptage terminé.")

100 personnages traités / 1026
200 personnages traités / 1026
300 personnages traités / 1026
400 personnages traités / 1026
500 personnages traités / 1026
600 personnages traités / 1026


In [10]:
# Assemblage du tableau final
output_df = names_df.drop(columns=["aliases"]).copy()

for col_name, values in count_data.items():
    output_df[col_name] = values

# Optionnel : total tous tomes confondus
book_cols = [f"book_{b}" for b in books]
output_df["total_occurrences"] = output_df[book_cols].sum(axis=1)

print(output_df.shape)
output_df.head(10)

(1027, 23)


,Noms humains,Second nom,Troisième nom,Quatrième nom,H/F/Na,ID unique,Morts,book_1,book_2,book_3,...,book_7,book_8,book_9,book_10,book_11,book_12,book_13,book_14,book_15,total_occurrences
0,Adessa,NaN,NaN,NaN,FEMME,6,True,0,0,0,...,4,0,0,0,0,0,0,0,0,4
1,Armald,NaN,NaN,NaN,HOMME,33,True,0,0,0,...,0,0,0,0,0,0,0,0,0,25
2,B’ner,NaN,NaN,NaN,HOMME,49,True,0,0,10,...,0,0,0,0,0,0,0,0,0,10
3,Barly,NaN,NaN,NaN,HOMME,67,True,0,0,0,...,0,0,0,0,0,0,0,0,0,3
4,Berchar,Berch,NaN,NaN,HOMME,88,True,0,0,0,...,0,0,0,0,0,0,0,0,0,41
5,Betrice,NaN,NaN,NaN,FEMME,96,True,0,0,0,...,73,0,0,0,0,0,0,0,0,73
6,Brare,NaN,NaN,NaN,HOMME,119,True,0,0,0,...,0,0,0,0,0,0,0,0,0,24
7,C’gan,NaN,NaN,NaN,HOMME,143,True,0,0,0,...,50,17,2,0,0,0,0,0,0,69
8,Caesar,Galliani,NaN,NaN,HOMME,151,True,14,0,0,...,0,0,0,0,0,0,0,0,0,14
9,Carola,NaN,NaN,NaN,FEMME,162,True,0,0,0,...,19,0,0,0,0,0,0,0,0,19


In [11]:
# Export
#output_df.to_excel(OUTPUT_XLSX, index=False)
output_df.to_csv(OUTPUT_CSV, index=False)

#print(f"Fichier Excel exporté : {OUTPUT_XLSX}")
print(f"Fichier CSV exporté   : {OUTPUT_CSV}")

Fichier CSV exporté   : pern_occurrences_par_tome.csv
